# N-LDF extension experiments: matched amount, threshold-tuned baseline, ablation

These are the experiments that the fixed-configuration benchmark cannot answer
(manuscript, Section 7): whether the N-LDF mechanism contributes anything
beyond the amount of synthetic data, and whether a tuned decision threshold
recovers the F1 gain of oversampling without resampling.

Everything is held identical to the archived run: the same 41 tasks, the same
partitions (`RepeatedStratifiedKFold` seeded with `stable_seed(dataset)`), the
same fold-wise preprocessing, the same four classifier policies and seeds, and
the same sampler code (`nldf.py`, hash-checked). Only the strategy list differs.
Every row also records the realized minority/majority ratio after resampling
and the wall-clock seconds of the resampling call (`SamplerSeconds`).

| Package | Strategies | What it isolates |
|---|---|---|
| `matched` | `N-LDF-r1.0`, `SMOTE-r0.75`, `ROS-r0.75`, `ROS-r1.0`, `Baseline-thr` | amount vs mechanism (with the archived `N-LDF` = r 0.75 and `SMOTE` = parity); threshold vs resampling |
| `ablation_statistic` | `ABL-lp0`, `ABL-lp1`, `ABL-localprop`, `ABL-uniform` | the support statistic itself |
| `ablation_components` | `ABL-nojitter`, `ABL-nodist`, `ABL-nosupport`, `ABL-noratio` | each pipeline component |
| `comparators` | `Baseline-cw`, `SLS`, `SVM-SMOTE`, `KMeans-SMOTE` | class weighting without resampling; the nearest safety-, boundary- and density-guided oversamplers at parity |
| `sensitivity` | `SENS-lp0.5`, `SENS-lp0.9`, `SENS-kd5`, `SENS-kd11`, `SENS-acc-0.1`, `SENS-acc+0.1`, `SENS-rb2.0` | one parameter moved from its archived value: lambda_p, the bandwidth neighbourhood, the acceptance threshold, the ratio bound |

Each package writes its own `fold_level_results.csv`; `extension_analysis.py`
merges them with the archived records and runs the same inference procedures.
Seeds: every N-LDF variant draws its sampler seed under the name `N-LDF`, and
`SMOTE-r0.75` under `SMOTE`, so a variant shares the archived random stream of
the strategy it modifies. `Baseline-thr` fits the classifier on the
unresampled training partition and chooses its decision threshold by inner
stratified cross-validation (3 folds) on that partition to maximise F1 of the
out-of-fold probabilities; the chosen threshold is logged. `Baseline-cw` fits
the classifier on the unresampled partition with balanced class weights
(`class_weight="balanced"`; `scale_pos_weight = n_maj / n_min` for XGBoost).
`SLS` is Safe-Level-SMOTE (Bunkhumpornpat et al., 2009), implemented in
`nldf_comparators.py` from the paper's pseudocode and self-tested; `SVM-SMOTE`
and `KMeans-SMOTE` are the imbalanced-learn implementations with the same
SMOTE fallback and audit record as the archived ADASYN.


### Cell 0: packages and code

In [ ]:
"""Cell 0: packages and repository code.

Run from the root of the n-ldf repository (locally), or let the cell clone it
(Colab). Nothing here mounts Google Drive; results go to ./nldf_results_41_v2_final,
the output directory name of the archived run.
"""
import os
import subprocess
import sys
from pathlib import Path

for pkg in ["imbalanced-learn", "xgboost", "scikit-posthocs", "requests", "tqdm"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

if not Path("nldf.py").exists():
    subprocess.check_call(["git", "clone", "-q", "https://github.com/ahmadilham-idsrg/n-ldf.git"])
    os.chdir("n-ldf")
print("working directory:", Path.cwd())
print("nldf.py present   :", Path("nldf.py").exists())


### Cell 1: imports, constants and the 41 KEEL tasks (verbatim)

In [ ]:
"""Cell 1 (REVISED): Import dan Constants dengan KDD Cup Fix"""

from __future__ import annotations

import argparse
import hashlib
import importlib.metadata
import io
import json
import pickle
import platform
import re
import signal
import sys
import time
import zipfile
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Optional

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from tqdm import tqdm
from urllib3.util.retry import Retry

warnings.filterwarnings('ignore')

# Optional dependencies
try:
    from imblearn.combine import SMOTETomek
    from imblearn.over_sampling import ADASYN, BorderlineSMOTE, SMOTE
except ImportError:
    SMOTETomek = ADASYN = BorderlineSMOTE = SMOTE = None

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

try:
    import scikit_posthocs as sp
except ImportError:
    sp = None

# Constants
RANDOM_STATE = 42
METHODS = ("Baseline", "SMOTE", "B-SMOTE", "ADASYN", "SMOTE-Tomek", "N-LDF")
CLASSIFIERS = ("RF", "SVM-Linear", "SVM-RBF", "XGBoost")
GLOBAL_TIMEOUT = 3600

# FIXED DATASETS: Remove problematic URLs, use alternative sources
DATASETS: dict[str, str] = {
    # Mild imbalance
    "glass1": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/glass1.zip",
    "pima": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/pima.zip",
    "iris0": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/iris0.zip",
    "yeast1": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast1.zip",
    "haberman": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/haberman.zip",
    "vehicle1": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/vehicle1.zip",
    "ecoli1": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/ecoli1.zip",
    "ecoli2": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/ecoli2.zip",
    "segment0": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/segment0.zip",
    "yeast3": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast3.zip",

    # High imbalance
    "yeast-2_vs_4": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast-2_vs_4.zip",
    "vowel0": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/vowel0.zip",
    "glass4": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/glass4.zip",
    "ecoli4": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/ecoli4.zip",
    "abalone9-18": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/abalone9-18.zip",
    "yeast-1-4-5-8_vs_7": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast-1-4-5-8_vs_7.zip",
    "glass5": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/glass5.zip",
    "yeast-2_vs_8": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast-2_vs_8.zip",
    "yeast4": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast4.zip",
    "yeast-1-2-8-9_vs_7": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast-1-2-8-9_vs_7.zip",
    "yeast5": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast5.zip",
    "ecoli-0-1-3-7_vs_2-6": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/ecoli-0-1-3-7_vs_2-6.zip",

    # Extreme imbalance
    "abalone-21_vs_8": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/abalone-21_vs_8.zip",
    "yeast6": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/yeast6.zip",
    "winequality-white-3_vs_7": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/winequality-white-3_vs_7.zip",
    "winequality-red-8_vs_6-7": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/winequality-red-8_vs_6-7.zip",
    "abalone-19_vs_10-11-12-13": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/abalone-19_vs_10-11-12-13.zip",
    "winequality-white-3-9_vs_5": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/winequality-white-3-9_vs_5.zip",
    "poker-8-9_vs_6": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/poker-8-9_vs_6.zip",
    "shuttle-2_vs_5": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/shuttle-2_vs_5.zip",
    "winequality-red-3_vs_5": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/winequality-red-3_vs_5.zip",
    "abalone-20_vs_8-9-10": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/abalone-20_vs_8-9-10.zip",
    "poker-8-9_vs_5": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/poker-8-9_vs_5.zip",
    "poker-8_vs_6": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/poker-8_vs_6.zip",
    "abalone19": "https://sci2s.ugr.es/keel/dataset/data/imbalanced/abalone19.zip",

    # FIXED: KDD Cup datasets dengan alternative processing flag
    "kddcup-land_vs_portsweep": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/kddcup-land_vs_portsweep.zip",
    "kddcup-buffer_overflow_vs_back": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/kddcup-buffer_overflow_vs_back.zip",
    "kddcup-land_vs_satan": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/kddcup-land_vs_satan.zip",
    "kddcup-rootkit-imap_vs_back": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/kddcup-rootkit-imap_vs_back.zip",
    "kr-vs-k-zero_vs_eight": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/kr-vs-k-zero_vs_eight.zip",
    "kr-vs-k-zero_vs_fifteen": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p3/kr-vs-k-zero_vs_fifteen.zip",
}

# Flag problematic datasets yang butuh special handling
PROBLEMATIC_DATASETS = {
    "kddcup-land_vs_portsweep", "kddcup-buffer_overflow_vs_back",
    "kddcup-land_vs_satan", "kddcup-rootkit-imap_vs_back",
    "kr-vs-k-zero_vs_eight", "kr-vs-k-zero_vs_fifteen"
}

print(f"✅ Cell 1 loaded: {len(DATASETS)} datasets configured")
print(f"⚠️  Problematic datasets (special handling): {len(PROBLEMATIC_DATASETS)}")

### Cell 2: dataset loading, preprocessing, classifiers (verbatim)

In [ ]:
"""Cell 2 (REVISED): Classes dengan Fixed Categorical Handling"""

@dataclass
class LoadedDataset:
    """Container for loaded dataset with metadata."""
    name: str
    X: pd.DataFrame
    y: np.ndarray
    numeric_columns: list[int]
    categorical_columns: list[int]
    n_samples: int
    n_features: int
    n_minority: int
    n_majority: int
    imbalance_ratio: float
    severity: str
    source_url: str
    sha256: str
    is_problematic: bool = False  # NEW: Flag for special handling


@dataclass
class CheckpointState:
    """State for checkpoint/resume functionality."""
    completed_datasets: set = field(default_factory=set)
    results: list[dict[str, object]] = field(default_factory=list)

    def save(self, path: Path) -> None:
        with open(path, 'wb') as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path: Path) -> 'CheckpointState':
        with open(path, 'rb') as f:
            return pickle.load(f)


class TimeoutError(Exception):
    pass


class FoldPreprocessor:
    """FOLD-PREPROCESSOR DENGAN FIX UNTUK KDD CUP DATASETS"""

    def __init__(self, numeric_columns: list[int], categorical_columns: list[int],
                 dataset_name: str = ""):
        self.numeric_columns = numeric_columns
        self.categorical_columns = categorical_columns
        self.dataset_name = dataset_name
        self.transformer: ColumnTransformer | None = None
        self.categorical_slices: list[slice] = []
        self.original_feature_order: list[int] = []
        self.has_categorical = len(categorical_columns) > 0
        self._fitted_categories: list = []

    def fit(self, X_train: pd.DataFrame) -> "FoldPreprocessor":
        """Fit preprocessor dengan safe categorical handling."""
        transformers: list[tuple[str, Pipeline, list[int]]] = []
        self.original_feature_order = []

        # Numeric pipeline
        if self.numeric_columns:
            numeric_pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ])
            transformers.append(("numeric", numeric_pipeline, self.numeric_columns))
            self.original_feature_order.extend(self.numeric_columns)

        # Categorical pipeline dengan FIX
        if self.categorical_columns:
            # Hanya proses kolom kategorikal yang valid
            valid_cat_cols = []
            for col in self.categorical_columns:
                if col < X_train.shape[1]:
                    n_unique = X_train.iloc[:, col].nunique()
                    # Skip jika terlalu banyak kategori (KDD Cup issue)
                    if n_unique < min(100, len(X_train) * 0.5):
                        valid_cat_cols.append(col)
                    else:
                        # Treat as numeric instead
                        if col not in self.numeric_columns:
                            self.numeric_columns.append(col)

            if valid_cat_cols:
                try:
                    encoder = OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                        dtype=float,
                        max_categories=50  # LIMIT categories untuk KDD Cup
                    )
                except TypeError:
                    encoder = OneHotEncoder(
                        handle_unknown="ignore",
                        sparse=False,
                        dtype=float
                    )

                categorical_pipeline = Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", encoder),
                ])
                transformers.append(("categorical", categorical_pipeline, valid_cat_cols))
                self.original_feature_order.extend(valid_cat_cols)
                self.categorical_columns = valid_cat_cols  # Update dengan yang valid saja

        if not transformers:
            # Fallback: semua kolom sebagai numerik
            all_cols = list(range(X_train.shape[1]))
            numeric_pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ])
            transformers.append(("numeric", numeric_pipeline, all_cols))
            self.numeric_columns = all_cols
            self.categorical_columns = []
            self.has_categorical = False

        self.transformer = ColumnTransformer(transformers, remainder="drop", sparse_threshold=0)
        self.transformer.fit(X_train)

        # Build categorical slices SAFELY
        self.categorical_slices = []
        if self.has_categorical and self.categorical_columns:
            try:
                if hasattr(self.transformer, 'transformers_'):
                    feature_idx = 0
                    for name, _, cols in self.transformer.transformers_:
                        if name == "numeric":
                            feature_idx += len(cols)
                        elif name == "categorical":
                            cat_transformer = self.transformer.named_transformers_["categorical"]
                            encoder = cat_transformer.named_steps["onehot"]
                            if hasattr(encoder, 'categories_'):
                                for categories in encoder.categories_:
                                    n_cats = len(categories)
                                    if n_cats > 0:  # Skip empty categories
                                        self.categorical_slices.append(
                                            slice(feature_idx, feature_idx + n_cats)
                                        )
                                        feature_idx += n_cats
            except Exception as e:
                # Jika gagal, treat all as numeric
                self.has_categorical = False
                self.categorical_slices = []

        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        """Transform data safely."""
        if self.transformer is None:
            raise RuntimeError("FoldPreprocessor must be fitted before transform")
        try:
            return np.asarray(self.transformer.transform(X), dtype=float)
        except Exception as e:
            # Fallback: hanya gunakan kolom numerik
            if self.numeric_columns:
                return np.asarray(X.iloc[:, self.numeric_columns], dtype=float)
            else:
                return np.asarray(X, dtype=float)

    def project_categorical(self, X: np.ndarray) -> np.ndarray:
        """FIXED: Project categorical dengan empty array handling."""
        if not self.has_categorical or not self.categorical_slices:
            return np.asarray(X, dtype=float)

        projected = np.asarray(X, dtype=float).copy()

        for block in self.categorical_slices:
            # SAFETY CHECKS
            if block.start >= projected.shape[1]:
                continue
            if block.stop > projected.shape[1]:
                continue

            values = projected[:, block]
            if values.size == 0 or values.shape[1] == 0:
                continue

            try:
                winners = np.argmax(values, axis=1)
                values[:] = 0.0
                values[np.arange(len(values)), winners] = 1.0
                projected[:, block] = values
            except (ValueError, IndexError):
                # Jika gagal, biarkan values apa adanya
                continue

        return projected


# Copy semua fungsi utility yang sama (timeout_handler, stable_seed, dll)
def timeout_handler(signum, frame):
    raise TimeoutError("Dataset processing timed out")

def stable_seed(*parts: object, base: int = RANDOM_STATE) -> int:
    payload = "|".join(map(str, (base, *parts))).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "little")

def severity_from_ir(ir: float) -> str:
    if ir < 9:
        return "Mild"
    if ir < 40:
        return "High"
    return "Extreme"

def make_http_session() -> requests.Session:
    retry = Retry(total=3, connect=3, read=3, backoff_factor=1.0,
                  status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=("GET",))
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.headers.update({"User-Agent": "NLDF-reproducibility-study/1.0"})
    return session

def parse_attribute_type(line: str) -> bool:
    lowered = line.lower()
    return bool(re.search(r"\b(real|integer|numeric)\b", lowered))

def effective_k(y_train: np.ndarray, requested: int = 5) -> int:
    n_minority = int((y_train == 1).sum())
    if n_minority < 2:
        raise ValueError("At least two training minority observations are required")
    return min(requested, n_minority - 1)

def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray) -> dict[str, float]:
    sensitivity = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    specificity = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    return {
        "F1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall": sensitivity,
        "Specificity": specificity,
        "G-Mean": float(np.sqrt(sensitivity * specificity)),
        "AUC-ROC": roc_auc_score(y_true, y_score),
        "AUPRC": average_precision_score(y_true, y_score),
    }

def build_classifiers(seed: int) -> dict[str, object]:
    if XGBClassifier is None:
        raise RuntimeError("xgboost is required")

    classifiers = {
        "RF": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed),
        "SVM-Linear": SVC(C=1.0, kernel="linear", probability=True, random_state=seed),
        "SVM-RBF": SVC(C=1.0, kernel="rbf", gamma="scale", probability=True, random_state=seed),
        "XGBoost": XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                  objective="binary:logistic", eval_metric="logloss",
                                  n_jobs=-1, random_state=seed),
    }

    # Warm-up XGBoost
    X_warm = np.random.randn(10, 4)
    y_warm = np.random.randint(0, 2, 10)
    classifiers["XGBoost"].fit(X_warm, y_warm)

    return classifiers


# UPDATED: load_keel_dataset dengan problem flag
def load_keel_dataset(name: str, url: str, session: requests.Session,
                      cache_dir: Path) -> LoadedDataset:
    """Load dataset dengan problem flag."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{name}.zip"
    payload: bytes | None = None

    if cache_path.exists():
        cached_payload = cache_path.read_bytes()
        if zipfile.is_zipfile(io.BytesIO(cached_payload)):
            payload = cached_payload

    if payload is None:
        response = session.get(url, timeout=60)
        response.raise_for_status()
        payload = response.content
        if not zipfile.is_zipfile(io.BytesIO(payload)):
            raise ValueError(f"{name}: URL did not return a valid ZIP archive")
        cache_path.write_bytes(payload)

    digest = hashlib.sha256(payload).hexdigest()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        dat_files = [m for m in archive.namelist() if m.lower().endswith(".dat")]
        if not dat_files:
            raise ValueError(f"{name}: ZIP archive contains no .dat file")
        with archive.open(dat_files[0]) as handle:
            content = None
            for encoding in ['utf-8', 'latin-1', 'cp1252']:
                try:
                    content = handle.read().decode(encoding)
                    break
                except UnicodeDecodeError:
                    continue
            if content is None:
                raise ValueError(f"{name}: Unable to decode file")

    lines = content.splitlines()
    try:
        data_index = next(i for i, line in enumerate(lines)
                        if line.strip().lower() == "@data")
    except StopIteration as exc:
        raise ValueError(f"{name}: @data marker not found") from exc

    attribute_lines = [line.strip() for line in lines[:data_index]
                      if line.strip().lower().startswith("@attribute")]

    raw_rows = [line.strip() for line in lines[data_index + 1:]
                if line.strip() and not line.lstrip().startswith(("%", "@"))]

    frame = pd.read_csv(io.StringIO("\n".join(raw_rows)), header=None,
                       sep=r",\s*", engine="python", dtype=str,
                       keep_default_na=False).replace(
        {"?": np.nan, "<null>": np.nan, "NULL": np.nan, "": np.nan})

    feature_types = [parse_attribute_type(line) for line in attribute_lines[:-1]]
    numeric_columns = [i for i, is_num in enumerate(feature_types) if is_num]
    categorical_columns = [i for i, is_num in enumerate(feature_types) if not is_num]

    X = frame.iloc[:, :-1].copy()
    for column in numeric_columns:
        X[column] = pd.to_numeric(X[column], errors="coerce")
    for column in categorical_columns:
        X[column] = X[column].where(X[column].notna(), np.nan).astype(object)

    target = frame.iloc[:, -1]
    valid_target = target.notna()
    X = X.loc[valid_target].reset_index(drop=True)
    target = target.loc[valid_target].astype(str).reset_index(drop=True)

    labels, counts = np.unique(target, return_counts=True)
    if len(labels) != 2:
        raise ValueError(f"{name}: expected binary labels, found {len(labels)}")

    minority_label = labels[np.argmin(counts)]
    y = (target.to_numpy() == minority_label).astype(int)
    n_minority = int(y.sum())
    n_majority = int(len(y) - n_minority)

    ir = n_majority / n_minority if n_minority > 0 else float('inf')

    is_problematic = name in PROBLEMATIC_DATASETS

    return LoadedDataset(
        name=name, X=X, y=y,
        numeric_columns=numeric_columns,
        categorical_columns=categorical_columns,
        n_samples=len(y), n_features=X.shape[1],
        n_minority=n_minority, n_majority=n_majority,
        imbalance_ratio=ir,
        severity=severity_from_ir(ir),
        source_url=url, sha256=digest,
        is_problematic=is_problematic
    )


class AuditedADASYN:
    """ADASYN with SMOTE fallback."""
    def __init__(self, n_neighbors: int, random_state: int):
        self.n_neighbors = n_neighbors
        self.random_state = random_state
        self.last_diagnostics_: dict[str, object] = {}

    def fit_resample(self, X: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        self.last_diagnostics_ = {"SamplerStatus": "success",
                                   "SamplerFallback": "none", "SamplerError": ""}
        try:
            return ADASYN(n_neighbors=self.n_neighbors,
                         random_state=self.random_state).fit_resample(X, y)
        except (RuntimeError, ValueError) as exc:
            message = str(exc)
            if ("not suited" in message or "No samples" in message):
                X_res, y_res = SMOTE(k_neighbors=self.n_neighbors,
                                    random_state=self.random_state).fit_resample(X, y)
                self.last_diagnostics_ = {"SamplerStatus": "fallback",
                                         "SamplerFallback": "SMOTE",
                                         "SamplerError": message}
                return X_res, y_res
            raise

print("✅ Cell 2 loaded: Fixed FoldPreprocessor for KDD Cup datasets")

### Cell 3: extension samplers (nldf.py hash-checked, nldf_ablation.py self-tested)

In [ ]:
"""Cell 3: samplers for the extension strategies.

nldf.py (hash-checked) provides the archived configuration; nldf_ablation.py
derives every variant from it by changing exactly one thing and self-tests
that its default configuration reproduces NLDFv2 bit for bit.
"""
import hashlib

import nldf
import nldf_ablation
import nldf_comparators
from nldf import NLDFv2
from nldf_ablation import build_variant, VARIANTS, MATCHED, ABLATION_STATISTIC, ABLATION_COMPONENTS, SENSITIVITY
from nldf_comparators import build_comparator, class_weighted, COMPARATORS, COMPARATORS_R075

NLDF_SHA256 = "77c5f72721508e0cf48ac92af162e53c784dabb516002e460f49b6d0fa2a9872"
_digest = hashlib.sha256(Path(nldf.__file__).read_bytes()).hexdigest()
if _digest != NLDF_SHA256:
    raise RuntimeError(f"nldf.py hash {_digest} differs from the archived implementation")
print(f"nldf.py verified: SHA-256 {_digest[:8]}...{_digest[-4:]}")
nldf_ablation.selftest(verbose=False)
print("nldf_ablation self-test passed (default flags == archived sampler)")
nldf_comparators.selftest(verbose=False)
print("nldf_comparators self-test passed")

from imblearn.over_sampling import RandomOverSampler

PACKAGES = {
    "matched": ["N-LDF-r1.0", "SMOTE-r0.75", "ROS-r0.75", "ROS-r1.0", "Baseline-thr"],
    "ablation_statistic": list(ABLATION_STATISTIC),
    "ablation_components": list(ABLATION_COMPONENTS),
    "comparators": list(COMPARATORS),
    "sensitivity": list(SENSITIVITY),
    # every other oversampler at the N-LDF target r = 0.75 (parity versions: archived run / comparators)
    "matched_all": list(COMPARATORS_R075),
}
# sampler seed aliases: a variant shares the random stream of the strategy it modifies;
# the SMOTE-family comparators share SMOTE's, so a recorded fallback equals the archived SMOTE call
SEED_ALIAS = {**{v: "N-LDF" for v in VARIANTS}, "SMOTE-r0.75": "SMOTE", "Baseline-thr": "Baseline",
              "Baseline-cw": "Baseline", "SLS": "SMOTE", "SVM-SMOTE": "SMOTE", "KMeans-SMOTE": "SMOTE",
              # r = 0.75 versions share the random stream of their parity version
              "B-SMOTE-r0.75": "B-SMOTE", "ADASYN-r0.75": "ADASYN", "SMOTE-Tomek-r0.75": "SMOTE-Tomek",
              "SLS-r0.75": "SMOTE", "SVM-SMOTE-r0.75": "SMOTE", "KMeans-SMOTE-r0.75": "SMOTE"}
THRESHOLD_TUNED = {"Baseline-thr"}
CLASS_WEIGHTED = {"Baseline-cw"}


def build_sampler_ext(method: str, y_train: np.ndarray,
                      projector: Callable[[np.ndarray], np.ndarray], seed: int):
    """Sampler for an extension strategy (None for the two baselines)."""
    k = effective_k(y_train)
    if method in ("Baseline", "Baseline-thr", "Baseline-cw"):
        return None
    if method in COMPARATORS or method in COMPARATORS_R075:
        return build_comparator(method, k, seed)
    if method == "SMOTE-r0.75":
        return SMOTE(sampling_strategy=0.75, k_neighbors=k, random_state=seed)
    if method == "ROS-r0.75":
        return RandomOverSampler(sampling_strategy=0.75, random_state=seed)
    if method == "ROS-r1.0":
        return RandomOverSampler(sampling_strategy=1.0, random_state=seed)
    if method in VARIANTS or method == "N-LDF":
        return build_variant(method, y_train, projector, seed)
    raise KeyError(f"Unknown extension strategy: {method}")


def tune_threshold(clf, X_train: np.ndarray, y_train: np.ndarray, seed: int,
                   n_splits: int = 3) -> float:
    """Decision threshold for the unresampled baseline, chosen on the training
    partition only: out-of-fold predicted probabilities from an inner
    stratified k-fold (k = min(3, n_min)), threshold maximising F1 over the
    unique probability values. Falls back to 0.5 when the minority has fewer
    than two training instances."""
    from sklearn.base import clone
    from sklearn.model_selection import StratifiedKFold
    n_min = int((y_train == 1).sum())
    k = min(n_splits, n_min)
    if k < 2:
        return 0.5
    oof = np.zeros(len(y_train))
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    for tr, va in skf.split(X_train, y_train):
        inner = clone(clf)
        inner.fit(X_train[tr], y_train[tr])
        oof[va] = inner.predict_proba(X_train[va])[:, 1]
    best_t, best_f1 = 0.5, -1.0
    for t in np.unique(oof):
        f1 = f1_score(y_train, (oof >= t).astype(int), pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = float(t), f1
    return best_t


def build_classifier_ext(clf_name: str, seed: int, y_res: np.ndarray, method: str):
    """The archived classifier policy; balanced class weights for the weighted baseline."""
    clf = build_classifiers(seed)[clf_name]
    if method in CLASS_WEIGHTED:
        clf = class_weighted(clf, y_res)
    return clf


_y = np.array([1] * 30 + [0] * 170)
for name in [m for pkg in PACKAGES.values() for m in pkg]:
    s = build_sampler_ext(name, _y, lambda X: X, 42)
    print(f"  {name:<20} -> {type(s).__name__ if s is not None else 'no resampling'}")


### Cell 4: benchmark loop (verbatim with the extension edits)

In [ ]:
"""Cell 4: the executed benchmark loop with the extension strategies.

Verbatim except for: the strategy list as a parameter, seed aliases, the
threshold-tuned baseline, and the realized-amount columns.
"""

def run_benchmark_ext(
    datasets: dict[str, LoadedDataset],
    n_splits: int,
    n_repeats: int,
    checkpoint_path: Optional[Path] = None,
    failed_datasets: Optional[set[str]] = None,
    skip_failures: bool = False,  # Default False for debugging
    methods: tuple = tuple(METHODS),
) -> tuple[pd.DataFrame, set[str]]:
    """Run benchmark dengan validasi ketat untuk N-LDF."""

    checkpoint = CheckpointState()
    if checkpoint_path and checkpoint_path.exists():
        checkpoint = CheckpointState.load(checkpoint_path)
        print(f"Resuming from checkpoint: {len(checkpoint.completed_datasets)} processed")

    new_failures = set()
    expected_per_dataset = n_splits * n_repeats * len(methods) * len(CLASSIFIERS)

    for dataset_name, dataset in tqdm(datasets.items(), desc="Datasets"):
        if dataset_name in checkpoint.completed_datasets:
            continue

        if failed_datasets and dataset_name in failed_datasets:
            continue

        # Adaptive folds
        if dataset.is_problematic:
            actual_folds = min(n_splits, max(2, dataset.n_minority // 5))
            actual_repeats = min(n_repeats, 2)
        elif dataset.n_minority < n_splits * 2:
            actual_folds = max(2, dataset.n_minority // 2)
            actual_repeats = n_repeats
        else:
            actual_folds = n_splits
            actual_repeats = n_repeats

        print(f"\n📊 {dataset_name}: folds={actual_folds}, repeats={actual_repeats}, "
              f"minority={dataset.n_minority}")

        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(GLOBAL_TIMEOUT)

        dataset_results_count = 0
        nldf_count = 0
        dataset_failed = False

        try:
            splitter = RepeatedStratifiedKFold(
                n_splits=actual_folds,
                n_repeats=actual_repeats,
                random_state=stable_seed(dataset_name),
            )

            for split_index, (train_idx, test_idx) in enumerate(
                splitter.split(dataset.X, dataset.y)
            ):
                repeat = split_index // actual_folds
                fold = split_index % actual_folds

                X_train_raw = dataset.X.iloc[train_idx]
                X_test_raw = dataset.X.iloc[test_idx]
                y_train = dataset.y[train_idx]
                y_test = dataset.y[test_idx]

                if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
                    continue

                preprocessor = FoldPreprocessor(
                    dataset.numeric_columns,
                    dataset.categorical_columns,
                    dataset_name=dataset_name
                ).fit(X_train_raw)

                X_train = preprocessor.transform(X_train_raw)
                X_test = preprocessor.transform(X_test_raw)

                if np.isnan(X_train).any() or np.isnan(X_test).any():
                    continue

                for method in methods:
                    seed = stable_seed(dataset_name, repeat, fold, SEED_ALIAS.get(method, method))

                    try:
                        sampler = build_sampler_ext(
                            method, y_train, preprocessor.project_categorical, seed
                        )

                        _t0 = time.perf_counter()
                        if sampler is None:
                            X_res, y_res = X_train.copy(), y_train.copy()
                            diag = {}
                        else:
                            X_res, y_res = sampler.fit_resample(X_train, y_train)
                            try:
                                X_res = preprocessor.project_categorical(X_res)
                            except Exception:
                                pass
                            diag = getattr(sampler, "last_diagnostics_", {})

                        sampler_seconds = time.perf_counter() - _t0

                        if len(np.unique(y_res)) < 2:
                            continue

                        # realized resampling amount (identical definition for every strategy)
                        n_min_after = int((y_res == 1).sum())
                        n_maj_after = int((y_res == 0).sum())
                        amount = {"NMinTrain": int((y_train == 1).sum()),
                                  "NMajTrain": int((y_train == 0).sum()),
                                  "NMinAfter": n_min_after, "NMajAfter": n_maj_after,
                                  "AchievedMinMajRatio": n_min_after / max(n_maj_after, 1),
                                  "SamplerSeconds": sampler_seconds}

                        for clf_name in CLASSIFIERS:
                            try:
                                clf = build_classifier_ext(
                                    clf_name, stable_seed(dataset_name, repeat, fold, clf_name), y_res, method
                                )

                                threshold = float("nan")
                                if method in THRESHOLD_TUNED:
                                    threshold = tune_threshold(clf, X_res, y_res,
                                                               stable_seed(dataset_name, repeat, fold, clf_name, "thr"))
                                clf.fit(X_res, y_res)
                                y_score = clf.predict_proba(X_test)[:, 1]
                                if method in THRESHOLD_TUNED:
                                    y_pred = (y_score >= threshold).astype(int)
                                else:
                                    y_pred = clf.predict(X_test)
                                metrics = evaluate_predictions(y_test, y_pred, y_score)

                                checkpoint.results.append({
                                    "Dataset": dataset_name,
                                    "Severity": dataset.severity,
                                    "IR": dataset.imbalance_ratio,
                                    "Repeat": repeat,
                                    "Fold": fold,
                                    "Method": method,
                                    "Classifier": clf_name,
                                    "NTrain": len(train_idx),
                                    "NTest": len(test_idx),
                                    "NResampled": len(y_res),
                                    "Threshold": threshold,
                                    **amount,
                                    **metrics,
                                    **diag,
                                })
                                dataset_results_count += 1
                                if method.startswith(("N-LDF", "ABL-", "SENS-")):
                                    nldf_count += 1

                            except Exception as e:
                                if method.startswith(("N-LDF", "ABL-", "SENS-")) and not skip_failures:
                                    print(f"\n❌ {method} + {clf_name} FAILED:")
                                    traceback.print_exc()
                                    raise  # Don't hide N-LDF errors!
                                continue

                    except Exception as e:
                        if method.startswith(("N-LDF", "ABL-", "SENS-")) and not skip_failures:
                            print(f"\n❌ {method} build/fit FAILED:")
                            traceback.print_exc()
                            raise  # Don't hide N-LDF errors!
                        continue

            # ============================================
            # VALIDASI SETELAH DATASET SELESAI
            # ============================================
            print(f"   Results: {dataset_results_count} rows, N-LDF variants: {nldf_count} rows")

            if dataset_results_count > 0:
                # Check N-LDF presence (only when the package contains an N-LDF variant)
                if nldf_count == 0 and any(m.startswith(("N-LDF", "ABL-", "SENS-")) for m in methods):
                    msg = f"N-LDF produced 0 results for {dataset_name}!"
                    print(f"   ❌ {msg}")
                    if not skip_failures:
                        raise RuntimeError(msg)
                    else:
                        new_failures.add(dataset_name)
                        continue

                checkpoint.completed_datasets.add(dataset_name)
                if checkpoint_path:
                    checkpoint.save(checkpoint_path)
            else:
                print(f"   ❌ No results at all!")
                if not skip_failures:
                    raise RuntimeError(f"No results for {dataset_name}")
                new_failures.add(dataset_name)

        except TimeoutError:
            print(f"   ⏰ Timeout")
            new_failures.add(dataset_name)
        except Exception as e:
            print(f"   ❌ Error: {e}")
            if not skip_failures:
                raise
            new_failures.add(dataset_name)
        finally:
            signal.alarm(0)

    return pd.DataFrame(checkpoint.results), new_failures


print("✅ Cell 4: run_benchmark_ext defined (extension strategies, threshold-tuned baseline, realized amounts)")

### Cell 5: dataset diagnostics (verbatim)

In [ ]:
"""Cell 4.5: Dataset Diagnostics - Memeriksa Dataset Sebelum Benchmark"""

def diagnose_datasets(loaded_datasets: dict[str, LoadedDataset]):
    """Check datasets for potential issues before benchmark."""
    print("="*60)
    print("📊 DATASET DIAGNOSTICS")
    print("="*60)

    problematic = []
    healthy = []

    for name, ds in loaded_datasets.items():
        issues = []

        # Check minority samples
        if ds.n_minority < 10:
            issues.append(f"⚠️  Very few minority samples: {ds.n_minority}")

        # Check imbalance ratio
        if ds.imbalance_ratio > 100:
            issues.append(f"⚠️  Extreme imbalance: {ds.imbalance_ratio:.1f}:1")
        elif ds.imbalance_ratio > 50:
            issues.append(f"⚡ High imbalance: {ds.imbalance_ratio:.1f}:1")

        # Check dimensionality
        if ds.n_features > 100:
            issues.append(f"📐 High dimensional: {ds.n_features} features")

        # Check sample size
        if ds.n_samples < 50:
            issues.append(f"📏 Very small dataset: {ds.n_samples} samples")

        # Check categorical features
        if len(ds.categorical_columns) > ds.n_features * 0.5:
            issues.append(f"🔤 Many categorical features: {len(ds.categorical_columns)}/{ds.n_features}")

        if issues:
            problematic.append((name, ds, issues))
            print(f"\n❌ {name}:")
            print(f"   Samples: {ds.n_samples} | Features: {ds.n_features}")
            print(f"   Minority: {ds.n_minority} | IR: {ds.imbalance_ratio:.1f} | Severity: {ds.severity}")
            for issue in issues:
                print(f"   {issue}")
        else:
            healthy.append(name)
            print(f"✅ {name}: OK (n={ds.n_samples}, IR={ds.imbalance_ratio:.1f}, min={ds.n_minority})")

    print(f"\n{'='*60}")
    print(f"📈 DIAGNOSTICS SUMMARY")
    print(f"{'='*60}")
    print(f"   ✅ Healthy datasets: {len(healthy)}/{len(loaded_datasets)}")
    print(f"   ⚠️  Potentially problematic: {len(problematic)}/{len(loaded_datasets)}")

    if problematic:
        print(f"\n   Problematic datasets:")
        for name, ds, issues in problematic:
            print(f"   - {name}: {len(issues)} issue(s) | IR={ds.imbalance_ratio:.1f} | Minority={ds.n_minority}")

    # Severity distribution
    severity_count = {}
    for ds in loaded_datasets.values():
        severity_count[ds.severity] = severity_count.get(ds.severity, 0) + 1

    print(f"\n   Severity distribution:")
    for severity in ["Mild", "High", "Extreme"]:
        count = severity_count.get(severity, 0)
        print(f"   - {severity}: {count} datasets")

    return problematic, healthy


def suggest_cv_params(datasets: dict[str, LoadedDataset], default_folds: int = 5):
    """Suggest CV parameters based on dataset characteristics."""
    print(f"\n{'='*60}")
    print(f"🔧 CV PARAMETER SUGGESTIONS")
    print(f"{'='*60}")

    min_minority = min(ds.n_minority for ds in datasets.values())
    max_minority = max(ds.n_minority for ds in datasets.values())
    avg_minority = np.mean([ds.n_minority for ds in datasets.values()])

    print(f"   Minority samples - Min: {min_minority}, Max: {max_minority}, Avg: {avg_minority:.0f}")

    if min_minority < default_folds * 2:
        suggested_folds = max(2, min_minority // 2)
        print(f"   ⚠️  Minimum minority samples ({min_minority}) too low for {default_folds}-fold CV")
        print(f"   💡 Suggestion: Use {suggested_folds}-fold CV for small datasets")
    else:
        print(f"   ✅ Minimum minority samples ({min_minority}) sufficient for {default_folds}-fold CV")

    # List datasets that need reduced folds
    needs_reduction = []
    for name, ds in datasets.items():
        if ds.n_minority < default_folds * 2:
            needs_reduction.append((name, ds.n_minority, max(2, ds.n_minority // 2)))

    if needs_reduction:
        print(f"\n   Datasets needing fold reduction:")
        for name, min_count, suggested in sorted(needs_reduction, key=lambda x: x[1]):
            print(f"   - {name:30s}: {min_count} minority → suggest {suggested} folds")


# Diagnosa dataset jika sudah di-load
if 'loaded' in locals() and loaded:
    print("🔍 Running dataset diagnostics...\n")
    problems, healthy = diagnose_datasets(loaded)
    suggest_cv_params(loaded)
else:
    print("ℹ️  No datasets loaded yet.")
    print("   Datasets will be automatically diagnosed during benchmark.")
    print("   Run Cell 6 to load and diagnose datasets.")

print(f"\n✅ Cell 4.5 completed: Diagnostics ready")

### Cell 6: run a package

In [ ]:
"""Cell 6: run one package on the archived partitions.

PACKAGE: "matched", "matched_all", "ablation_statistic", "ablation_components", "comparators" or "sensitivity".
Each package has its own output directory and checkpoint, so it can be run in
separate Colab sessions and resumed. SMOKE_TEST restricts the run to the three
smoke-test tasks with two repeats.
"""
import shutil
from datetime import datetime

PACKAGE = "matched"
SMOKE_TEST = False
CLEAN_START = False
SMOKE_DATASETS = ["glass1", "winequality-white-3_vs_7", "kddcup-land_vs_portsweep"]

RUN_CONFIG = {
    "n_folds": 5,
    "n_repeats": 2 if SMOKE_TEST else 5,
    "methods": tuple(PACKAGES[PACKAGE]),
    "output_dir": Path(("smoke_" if SMOKE_TEST else "") + f"nldf_results_41_{PACKAGE}"),
    "cache_dir": Path("keel_cache"),
    "checkpoint_path": Path(("smoke_" if SMOKE_TEST else "") + f"checkpoint_{PACKAGE}.pkl"),
    "skip_failures": True,
    "timeout_per_dataset": 3600,
}
print("=" * 80)
print(f"🚀 EXTENSION PACKAGE '{PACKAGE}': {RUN_CONFIG['methods']}")
print(f"   CV: {RUN_CONFIG['n_folds']}-fold x {RUN_CONFIG['n_repeats']} repeats; classifiers {CLASSIFIERS}")
print("=" * 80)

if CLEAN_START:
    for path in [RUN_CONFIG["output_dir"], RUN_CONFIG["checkpoint_path"]]:
        if path.exists():
            shutil.rmtree(path) if path.is_dir() else path.unlink()
            print(f"  Deleted: {path}")

session = make_http_session()
loaded, load_failures = {}, []
DATASETS_TO_RUN = ({k: DATASETS[k] for k in SMOKE_DATASETS} if SMOKE_TEST else DATASETS)
for name, url in tqdm(DATASETS_TO_RUN.items(), desc="Loading datasets"):
    try:
        loaded[name] = load_keel_dataset(name, url, session, RUN_CONFIG["cache_dir"])
    except Exception as e:
        load_failures.append({"Dataset": name, "Error": str(e)[:150]})
        print(f"  ❌ {name}: {str(e)[:80]}")
print(f"✅ Loaded: {len(loaded)}/{len(DATASETS_TO_RUN)} datasets")

RUN_CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
pd.DataFrame([{"name": ds.name, "n_samples": ds.n_samples, "n_features": ds.n_features,
               "n_minority": ds.n_minority, "n_majority": ds.n_majority,
               "imbalance_ratio": round(ds.imbalance_ratio, 2), "severity": ds.severity,
               "is_problematic": ds.is_problematic, "sha256": ds.sha256}
              for ds in loaded.values()]).to_csv(RUN_CONFIG["output_dir"] / "dataset_audit.csv", index=False)

GLOBAL_TIMEOUT = RUN_CONFIG["timeout_per_dataset"]
start_time = datetime.now()
print(f"\n🚀 Start: {start_time:%Y-%m-%d %H:%M:%S}  (checkpoint: {RUN_CONFIG['checkpoint_path']})")
results, new_failures = run_benchmark_ext(
    loaded, RUN_CONFIG["n_folds"], RUN_CONFIG["n_repeats"],
    checkpoint_path=RUN_CONFIG["checkpoint_path"],
    failed_datasets=set(f["Dataset"] for f in load_failures),
    skip_failures=RUN_CONFIG["skip_failures"],
    methods=RUN_CONFIG["methods"],
)
elapsed = datetime.now() - start_time
results_path = RUN_CONFIG["output_dir"] / "fold_level_results.csv"
results.to_csv(results_path, index=False)
print(f"\n💾 {results_path}: {len(results)} rows in {elapsed}")
print("Strategies:", sorted(results["Method"].unique()) if len(results) else [])
if len(results):
    print(results.groupby("Method")[["F1", "Recall", "Specificity", "AUPRC", "AchievedMinMajRatio"]].mean().round(4))
if new_failures:
    print("Failed datasets:", sorted(new_failures))
print("\nTo download:  !zip -r extension_results.zip", RUN_CONFIG["output_dir"])


### Cell 7: analysis

In [ ]:
"""Cell 7: analysis of the extension results together with the archived records.

extension_analysis.py merges results/fold_level_results.csv (archived) with the
package outputs and runs the same aggregation and inference procedures as the
manuscript (dataset-level means, Wilcoxon-Pratt with Holm, source-cluster
bootstrap). Run it after every package you have, e.g.:
"""
import subprocess
import sys

archived = Path("results/fold_level_results.csv")
packages = [p for p in ["nldf_results_41_matched", "nldf_results_41_ablation_statistic",
                        "nldf_results_41_ablation_components", "nldf_results_41_comparators",
                        "nldf_results_41_sensitivity"]
            if Path(p, "fold_level_results.csv").exists()]
if archived.exists() and packages:
    cmd = [sys.executable, "extension_analysis.py", "--archived", str(archived),
           "--outdir", "results/extension"] + [f"{p}/fold_level_results.csv" for p in packages]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=False)
else:
    print("nothing to analyse yet: need results/fold_level_results.csv and at least one package directory")
